# 6.8 Random Forest

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.08_random-forest.ipynb)

Companion code for Section 6.8 of the book. Run the cells in order from top to bottom.

**Example 6.8.1**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

# ── Effect of n_estimators ────────────────────────────────────────
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
results = []
for n in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    results.append({'n': n, 'train': rf.score(X_train, y_train),
                    'test': rf.score(X_test, y_test), 'oob': rf.oob_score_})
    print(f'n={n:>4}: train={rf.score(X_train,y_train):.4f}  test={rf.score(X_test,y_test):.4f}  OOB={rf.oob_score_:.4f}')

# ── Best Random Forest ────────────────────────────────────────────
rf_best = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',    # m = √p features per split
    max_depth=None,         # Grow full trees
    min_samples_split=2,
    min_samples_leaf=1,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_best.fit(X_train, y_train)
print(f'\nBest RF: Test={rf_best.score(X_test,y_test):.4f}  OOB={rf_best.oob_score_:.4f}')

# Cross-validation
cv = cross_val_score(rf_best, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'5-Fold AUC: {cv.mean():.4f} ± {cv.std():.4f}')

# ── Feature importance ────────────────────────────────────────────
importance = rf_best.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf_best.estimators_], axis=0)
top = np.argsort(importance)[::-1][:10]

print('\nTop 10 Feature Importances (with std across trees):')
for i in top:
    print(f'  {data.feature_names[i]:30s}: {importance[i]:.4f} ± {std[i]:.4f}')

# ── Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Random Forest Analysis', fontsize=14, fontweight='bold')

# Accuracy vs n_estimators
ax = axes[0]
ns = [r['n'] for r in results]
ax.plot(ns, [r['train'] for r in results], 'b-o', ms=5, lw=2, label='Train')
ax.plot(ns, [r['test']  for r in results], 'r-o', ms=5, lw=2, label='Test')
ax.plot(ns, [r['oob']   for r in results], 'g-o', ms=5, lw=2, label='OOB')
ax.set_xlabel('Number of Trees'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Number of Trees'); ax.legend(); ax.grid(alpha=0.3)

# Feature importance with error bars
ax = axes[1]
ax.barh(range(10), importance[top][::-1], xerr=std[top][::-1],
       color='#1976D2', alpha=0.8, edgecolor='white', capsize=3)
ax.set_yticks(range(10))
ax.set_yticklabels([data.feature_names[i] for i in top[::-1]], fontsize=9)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Feature Importances ± Std (across 200 trees)')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('random_forest.png', dpi=150, bbox_inches='tight')
plt.show()